# TKAN Signal Evaluation

Load walk-forward predictions from any TKAN version, convert to a binary direction signal,
optionally layer the IVol z-score hedge exit, and backtest against LVC long-only.

**Switch models** by changing `MODEL` in the Config cell.  
**Toggle IVol filter** with `USE_IVOL_FILTER`.

In [11]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# MODEL: which prediction source to use
#   'v3'      → loads pred_cache.pkl  (5-day log-return predictions, full history)
#   'v2'      → expects a predictions CSV at PREDICTIONS_CSV_PATH
#   'custom'  → expects a parquet/csv at PREDICTIONS_CSV_PATH with columns:
#               [date, pred_return]  where pred_return > 0 → long
MODEL = 'v3'

# Path to a saved predictions file (for MODEL = 'v2' or 'custom')
PREDICTIONS_CSV_PATH = 'tkan/v2/predictions.csv'

# Signal threshold: long when predicted cumulative return > SIGNAL_THRESHOLD
SIGNAL_THRESHOLD = 0.0

# IVol hedge: exit to cash when IVol z-score >= IVOL_THRESHOLD
USE_IVOL_FILTER   = True
IVOL_WINDOW       = 126    # rolling window in days for z-score
IVOL_THRESHOLD    = 1.0    # z-score level that triggers exit
SIGNAL_LAG        = 1      # days: signal at close[T] → trade at open[T+1]

# Backtest window
START_DATE = '2015-01-01'
END_DATE   = None          # None = use all available data

# Paths (relative to this notebook's directory)
import os, sys
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
BTEST_ROOT   = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
PRED_CACHE   = os.path.join(NOTEBOOK_DIR, 'tkan', 'v3', 'weights', 'pred_cache.pkl')
LVC_CSV      = os.path.join(BTEST_ROOT, 'data', 'lvc_ohlcv.csv')

# Add sfera-db to path if not installed
for _p in [os.path.join(BTEST_ROOT, '..', 'sfera-db'), os.path.join(BTEST_ROOT, '..', '..', 'sfera-db')]:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))
# ──────────────────────────────────────────────────────────────────────────────

In [12]:
import pickle
import warnings
import numpy  as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

## 1 — Load LVC price data

In [13]:
lvc = pd.read_csv(LVC_CSV, parse_dates=['date'], index_col='date').sort_index()
lvc.index = pd.to_datetime(lvc.index).tz_localize(None)

if START_DATE:
    lvc = lvc.loc[START_DATE:]
if END_DATE:
    lvc = lvc.loc[:END_DATE]

# Daily log returns (used for backtest P&L)
lvc['log_ret'] = np.log(lvc['close'] / lvc['close'].shift(1))

print(f'LVC loaded: {lvc.index[0].date()} → {lvc.index[-1].date()}  ({len(lvc)} rows)')
lvc.tail(3)

LVC loaded: 2015-01-02 → 2026-03-20  (2872 rows)


,open,high,low,close,volume,log_ret
date,,,,,,
2026-03-18,41.6000,42.0450,40.7950,41.0650,234514,-0.0016
2026-03-19,39.9550,40.2300,39.0750,39.4250,623534,-0.0408
2026-03-20,40.0000,40.1350,37.9500,37.9850,654796,-0.0372


## 2 — Load model predictions

In [14]:
if MODEL == 'v3':
    with open(PRED_CACHE, 'rb') as f:
        pred_df, retrain_dates, cfg = pickle.load(f)
    pred_df.index = pd.to_datetime(pred_df.index).tz_localize(None)
    pred_series = pred_df.sum(axis=1).rename('pred_cumret')

elif MODEL in ('v2', 'custom'):
    _path = os.path.join(NOTEBOOK_DIR, PREDICTIONS_CSV_PATH)
    _ext  = os.path.splitext(_path)[1]
    if _ext == '.parquet':
        _df = pd.read_parquet(_path)
    else:
        _df = pd.read_csv(_path, parse_dates=['date'])
    _df = _df.set_index('date').sort_index()
    _df.index = pd.to_datetime(_df.index).tz_localize(None)
    pred_series = _df['pred_return'].rename('pred_cumret')

else:
    raise ValueError(f'Unknown MODEL: {MODEL}')

if START_DATE:
    pred_series = pred_series.loc[START_DATE:]
if END_DATE:
    pred_series = pred_series.loc[:END_DATE]

print(f'[{MODEL}] predictions: {pred_series.index[0].date()} → {pred_series.index[-1].date()}  ({len(pred_series)} rows)')

[v3] predictions: 2015-01-02 → 2026-03-13  (2866 rows)


## 3 — Load IVol data (optional)

In [15]:
ivol_series = None

if USE_IVOL_FILTER:
    try:
        import sfera_db
        _iv = sfera_db.query(
            'SELECT trade_date AS date, "3m_50d_ivol" AS ivol '
            "FROM bbgidx.index_implied_vol WHERE ticker = 'CAC' ORDER BY trade_date"
        )
        _iv['date'] = pd.to_datetime(_iv['date']).dt.tz_localize(None)
        ivol_series = _iv.set_index('date')['ivol'].sort_index()
        print(f'IVol: sfera  {ivol_series.index[0].date()} → {ivol_series.index[-1].date()}')
    except Exception as e:
        _fallback = os.path.join(BTEST_ROOT, '..', 'individual_strategies', 'ETF TKAN', 'Data', 'cac_ivol.csv')
        if os.path.exists(_fallback):
            _df = pd.read_csv(_fallback, parse_dates=[0], index_col=0)
            _df.index = pd.to_datetime(_df.index).tz_localize(None)
            ivol_series = _df.iloc[:, 0].rename('ivol').sort_index()
            print(f'IVol: CSV fallback  {ivol_series.index[0].date()} → {ivol_series.index[-1].date()}')
        else:
            print('⚠️  No IVol data — running without filter')
            USE_IVOL_FILTER = False

IVol: sfera  2007-01-02 → 2026-03-20


## 4 — Build signals

In [16]:
base_signal = (pred_series > SIGNAL_THRESHOLD).astype(float).rename('base_signal')

if USE_IVOL_FILTER and ivol_series is not None:
    rm = ivol_series.rolling(IVOL_WINDOW).mean()
    rs = ivol_series.rolling(IVOL_WINDOW).std()
    ivol_zscore = (ivol_series - rm) / rs
    ivol_filter = (ivol_zscore < IVOL_THRESHOLD).fillna(True).astype(float).rename('ivol_filter')
else:
    ivol_zscore = pd.Series(dtype=float)
    ivol_filter = pd.Series(1.0, index=base_signal.index, name='ivol_filter')

combined = (base_signal * ivol_filter.reindex(base_signal.index).ffill().fillna(1.0)).rename('combined_signal')
sig = combined.reindex(lvc.index).ffill().shift(SIGNAL_LAG).fillna(0.0)

# Master aligned DataFrame — single index = lvc trading days
master = lvc[['close']].copy()
master['pred_cumret'] = pred_series.reindex(lvc.index).ffill()
master['signal']      = sig
master['base_signal'] = base_signal.reindex(lvc.index).ffill().fillna(0.0)
master['ivol_zscore'] = ivol_zscore.reindex(lvc.index).ffill() if len(ivol_zscore) else np.nan
master = master.dropna(subset=['pred_cumret'])

pct_in = (sig > 0).mean()
print(f'Signal: {(sig>0).sum()} long / {len(sig)} days  ({pct_in:.1%} in market)  |  coverage {master.index[0].date()} → {master.index[-1].date()}')

Signal: 1866 long / 2872 days  (65.0% in market)  |  coverage 2015-01-02 → 2026-03-20


## 5 — Backtest

In [17]:
def run_backtest(price_series: pd.Series, signal: pd.Series, label: str) -> pd.DataFrame:
    """Vectorised long-only backtest. Returns daily P&L DataFrame."""
    log_ret = np.log(price_series / price_series.shift(1))
    strat   = (signal * log_ret).fillna(0.0)
    bh      = log_ret.fillna(0.0)
    return pd.DataFrame({
        f'{label}':      strat.cumsum().apply(np.exp) - 1,
        'Buy & Hold': bh.cumsum().apply(np.exp) - 1,
        f'{label}_daily': strat,
        'bh_daily': bh,
    })

results = run_backtest(lvc['close'], sig, label=f'TKAN {MODEL}')
#results.tail(3)

## 6 — Performance statistics

In [18]:
from scipy.stats import spearmanr

def stats(daily_log_ret: pd.Series, label: str) -> dict:
    r = daily_log_ret.dropna()
    ann   = 252
    cagr  = np.exp(r.sum() * ann / len(r)) - 1
    vol   = r.std() * np.sqrt(ann)
    sharpe = (r.mean() / r.std()) * np.sqrt(ann) if r.std() > 0 else 0
    cum   = r.cumsum().apply(np.exp)
    dd    = (cum / cum.cummax() - 1).min()
    calmar = cagr / abs(dd) if dd != 0 else 0
    return {'Strategy': label, 'CAGR': f'{cagr:.1%}', 'Vol': f'{vol:.1%}',
            'Sharpe': f'{sharpe:.2f}', 'MaxDD': f'{dd:.1%}', 'Calmar': f'{calmar:.2f}'}

# Direction accuracy (hit rate)
_strat_col = f'TKAN {MODEL}_daily'
_lvc_ret   = np.log(lvc['close'] / lvc['close'].shift(1))
_aligned   = pd.concat([sig.rename('signal'), _lvc_ret.rename('actual')], axis=1).dropna()
_long_days = _aligned[_aligned['signal'] > 0]
hit_rate   = (_long_days['actual'] > 0).mean()

# IC on raw prediction vs next-day return
_ic_df  = pd.concat([pred_series, _lvc_ret.shift(-1)], axis=1).dropna()
ic_val, _  = spearmanr(_ic_df.iloc[:, 0], _ic_df.iloc[:, 1])

perf = pd.DataFrame([
    stats(results[_strat_col], f'TKAN {MODEL}' + (' + IVol filter' if USE_IVOL_FILTER else '')),
    stats(results['bh_daily'],  'Buy & Hold LVC'),
])
perf['HitRate']  = [f'{hit_rate:.1%}', '-']
perf['IC (rank)'] = [f'{ic_val:.3f}', '-']
perf.set_index('Strategy', inplace=True)
perf

,CAGR,Vol,Sharpe,MaxDD,Calmar,HitRate,IC (rank)
Strategy,,,,,,,
TKAN v3 + IVol filter,7.0%,24.4%,0.28,-44.0%,0.16,53.4%,-0.003
Buy & Hold LVC,11.6%,36.9%,0.30,-63.9%,0.18,-,-


## 7 — Charts

In [19]:
import importlib, sys, os as _os
for _sp in [
    _os.path.join(BTEST_ROOT, '..', 'signum'),
    _os.path.join(BTEST_ROOT, '..', '..', 'signum'),
]:
    if _os.path.isdir(_sp) and _sp not in sys.path:
        sys.path.insert(0, _os.path.abspath(_sp))

import signum.engine.chart, signum.engine.dashboard, signum
importlib.reload(signum.engine.chart)
importlib.reload(signum.engine.dashboard)
from signum import Chart, Dashboard

strat_label = f'TKAN {MODEL}' + (' + IVol' if USE_IVOL_FILTER else '')

# All series sliced from `master` — same DatetimeIndex, no misalignment
_m = master.reset_index()
_m = _m.rename(columns={'date': 'Date', 'index': 'Date'})

# Add equity curves (indexed to 1.0 at start)
_log_ret  = np.log(_m['close'] / _m['close'].shift(1)).fillna(0.0)
_strat_dr = (_m['signal'].shift(1).fillna(0.0) * _log_ret)
_m['equity_strat'] = np.exp(_strat_dr.cumsum())
_m['equity_bh']    = np.exp(_log_ret.cumsum())

# ── Pane 1: LVC price line + blue shading where signal is long ───────────────
pane_price = (
    Chart(height=300)
    .line(_m[['Date', 'close']].rename(columns={'close': 'value'}),
          name='LVC', color='#b0c4de', value_col='value')
    .shade(_m[['Date', 'signal']].rename(columns={'signal': 'position'}),
           position_col='position', color='#1fa8e0', opacity=0.15)
)

# ── Pane 2: Raw NN prediction (5d cumulative log-return) ─────────────────────
pane_pred = (
    Chart(height=140)
    .baseline(_m[['Date', 'pred_cumret']].rename(columns={'pred_cumret': 'value'}),
              base_value=float(SIGNAL_THRESHOLD), value_col='value')
)

# ── Pane 3: Signal (0/1) + IVol z-score ──────────────────────────────────────
pane_signal = (
    Chart(height=110)
    .area(_m[['Date', 'signal']].rename(columns={'signal': 'value'}),
          name='Signal (long=1)', color='#50c878')
)
if USE_IVOL_FILTER and master['ivol_zscore'].notna().any():
    pane_signal = pane_signal.line(
        _m[['Date', 'ivol_zscore']].rename(columns={'ivol_zscore': 'value'}),
        name=f'IVol z-score  (exit ≥ {IVOL_THRESHOLD})', color='#e05050', width=1,
    )

# ── Pane 4: Equity curve — strategy vs buy & hold ────────────────────────────
_s = perf.iloc[0]
_b = perf.iloc[1]
_total_ret = f"{(_m['equity_strat'].iloc[-1] - 1):.1%}"
_bh_total  = f"{(_m['equity_bh'].iloc[-1] - 1):.1%}"
pane_equity = (
    Chart(height=200)
    .line(_m[['Date', 'equity_strat']].rename(columns={'equity_strat': 'value'}),
          name=strat_label, color='#1fa8e0', value_col='value')
    .line(_m[['Date', 'equity_bh']].rename(columns={'equity_bh': 'value'}),
          name='Buy & Hold', color='#888888', value_col='value')
    .stats_legend({
        'Strategy':    strat_label,
        'Total Ret':   _total_ret,
        'CAGR':        _s['CAGR'],
        'Sharpe':      _s['Sharpe'],
        'Max DD':      _s['MaxDD'],
        'Hit Rate':    _s['HitRate'],
        '──────────':  '',
        'B&H Total':   _bh_total,
        'B&H CAGR':    _b['CAGR'],
        'B&H Sharpe':  _b['Sharpe'],
    }, position='top-left')
)

# ── Dashboard ─────────────────────────────────────────────────────────────────
dash = Dashboard(
    panes=[pane_price, pane_pred, pane_signal, pane_equity],
    titles=[
        f'LVC  |  shaded = in market  |  {strat_label}',
        f'NN prediction [{MODEL}]  (threshold={SIGNAL_THRESHOLD})',
        f'Signal  +  IVol z-score  (exit ≥ {IVOL_THRESHOLD})' if USE_IVOL_FILTER else 'Signal',
        f'Equity curve  (normalised to 1.0)',
    ],
    theme='dark',
    gap=3,
)
dash


## 8 — Signal decomposition (base vs IVol filter)

In [20]:
# Quick breakdown: how many days did the IVol filter override the base signal?
if USE_IVOL_FILTER and ivol_series is not None:
    _b = base_signal.reindex(lvc.index).fillna(0)
    _f = ivol_filter.reindex(lvc.index).fillna(1)
    _both   = (_b > 0) & (_f > 0)
    _ivol_blocked = (_b > 0) & (_f == 0)
    print(f'Base signal long:         {(_b>0).sum():>5d} days')
    print(f'IVol filter active (exit):{(_f==0).sum():>5d} days')
    print(f'IVol blocked a long:      {_ivol_blocked.sum():>5d} days  ({_ivol_blocked.sum()/(_b>0).sum():.1%} of longs filtered)')
    print(f'Final long days:          {_both.sum():>5d} days')
else:
    print('IVol filter not active.')

Base signal long:          2243 days
IVol filter active (exit):  504 days
IVol blocked a long:        378 days  (16.9% of longs filtered)
Final long days:           1865 days
